In [9]:
import pandas as pd
import os

# ============================================================
# 1. LOAD DATA
# ============================================================

file_name = "2020_al_data_kaggle_upload_new_old_syllabi.csv"

# Get the folder where this Python file is located
# In Colab, __file__ is not defined. We can assume the file is in the current working directory.
# script_folder = os.path.dirname(os.path.abspath(__file__))

# Create full path to CSV
# Assuming the file is in the default Colab content directory
file_path = os.path.join('/content/', file_name)

# Read CSV
data = pd.read_csv(file_path)


# ============================================================
# 2. BASIC INFORMATION
# ============================================================

print("=" * 60)
print("DATA PROFILING REPORT")
print("=" * 60)

print("\nDataset:")
print(file_name)

print("\nNumber of Rows:", data.shape[0])
print("Number of Columns:", data.shape[1])


# ============================================================
# 3. COLUMN INFORMATION
# ============================================================

print("\n" + "=" * 60)
print("COLUMN INFORMATION")
print("=" * 60)

print(data.info())


# ============================================================
# 4. COLUMN NAMES
# ============================================================

print("\n" + "=" * 60)
print("COLUMN NAMES")
print("=" * 60)

for i, column in enumerate(data.columns, start=1):
    print(f"{i}. {column}")


# ============================================================
# 5. DATA TYPES
# ============================================================

print("\n" + "=" * 60)
print("DATA TYPES")
print("=" * 60)

print(data.dtypes)


# ============================================================
# 6. MISSING VALUES
# ============================================================

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing = data.isnull().sum()

missing_percentage = (missing / len(data)) * 100

missing_report = pd.DataFrame({
    "Missing Values": missing,
    "Missing Percentage": missing_percentage
})

print(missing_report)


# ============================================================
# 7. DUPLICATE RECORDS
# ============================================================

print("\n" + "=" * 60)
print("DUPLICATE RECORDS")
print("=" * 60)

duplicate_count = data.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)


# ============================================================
# 8. UNIQUE VALUES
# ============================================================

print("\n" + "=" * 60)
print("UNIQUE VALUES")
print("=" * 60)

for column in data.columns:
    print(f"{column}: {data[column].nunique()} unique values")


# ============================================================
# 9. NUMERICAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("NUMERICAL DATA SUMMARY")
print("=" * 60)

print(data.describe())


# ============================================================
# 10. CATEGORICAL DATA SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("CATEGORICAL DATA SUMMARY")
print("=" * 60)

categorical_columns = data.select_dtypes(
    include=["object", "category"]
).columns

for column in categorical_columns:

    print("\n" + "-" * 50)
    print("Column:", column)
    print("-" * 50)

    print(data[column].value_counts(dropna=False).head(20))


# ============================================================
# 11. MINIMUM AND MAXIMUM VALUES
# ============================================================

print("\n" + "=" * 60)
print("MINIMUM AND MAXIMUM VALUES")
print("=" * 60)

numeric_columns = data.select_dtypes(
    include=["number"]
).columns

for column in numeric_columns:

    print(
        f"{column}: "
        f"Min = {data[column].min()}, "
        f"Max = {data[column].max()}"
    )


# ============================================================
# 12. SAMPLE DATA
# ============================================================

print("\n" + "=" * 60)
print("FIRST 10 RECORDS")
print("=" * 60)

print(data.head(10))


# ============================================================
# 13. DATA QUALITY CHECK
# ============================================================

print("\n" + "=" * 60)
print("DATA QUALITY CHECK")
print("=" * 60)

print("\nColumns containing missing values:")

for column in data.columns:

    missing_count = data[column].isnull().sum()

    if missing_count > 0:
        print(
            f"- {column}: "
            f"{missing_count} missing values "
            f"({missing_count / len(data) * 100:.2f}%)"
        )


# ============================================================
# 14. PROFILE COMPLETE
# ============================================================

print("\n" + "=" * 60)
print("DATA PROFILING COMPLETED")
print("=" * 60)

DATA PROFILING REPORT

Dataset:
2020_al_data_kaggle_upload_new_old_syllabi.csv

Number of Rows: 337553
Number of Columns: 19

COLUMN INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 337553 entries, 0 to 337552
Data columns (total 19 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   index          337553 non-null  int64 
 1   stream         337553 non-null  object
 2   Zscore         337553 non-null  object
 3   district_rank  337553 non-null  object
 4   island_rank    337553 non-null  object
 5   al_year        337553 non-null  int64 
 6   sub1           337553 non-null  object
 7   sub1_r         337553 non-null  object
 8   sub2           337553 non-null  object
 9   sub2_r         337553 non-null  object
 10  sub3           337553 non-null  object
 11  sub3_r         337553 non-null  object
 12  cgt_r          337553 non-null  object
 13  ge_r           337553 non-null  object
 14  syllabus       337553 non-null  obj

The profiling report shows that `Zscore`, `district_rank`, `island_rank`, and several other 'rank' related columns are of `object` type, but they seem to represent numerical values. Let's inspect these columns to understand why they are not numeric and clean them.

In [10]:
print('Unique values for Zscore column:')
print(data['Zscore'].unique())

print('\nUnique values for district_rank column:')
print(data['district_rank'].unique())

print('\nUnique values for island_rank column:')
print(data['island_rank'].unique())

Unique values for Zscore column:
['-.3550' '-.2648' '-.4760' ... '-.7751' '-1.4303' '-1.6642']

Unique values for district_rank column:
['4336 (NEW)' '4154 (NEW)' '6910 (NEW)' ... '569 (OLD)' '512 (OLD)'
 '545 (OLD)']

Unique values for island_rank column:
['64994 (NEW)' '62338 (NEW)' '37307 (NEW)' ... '4787 (OLD)' '4438 (OLD)'
 '5184 (OLD)']


In [11]:
# ============================================================
# 13. TOP 10 HIGHEST Z-SCORES FOR EACH STREAM
# ============================================================

print("\n" + "=" * 60)
print("TOP 10 HIGHEST Z-SCORES FOR EACH STREAM")
print("=" * 60)

# Remove rows where stream or Zscore is missing
zscore_data = data.dropna(subset=["stream", "Zscore"])

# Group by stream and get the 10 highest Z-scores
top_10 = (
    zscore_data
    .sort_values(["stream", "Zscore"], ascending=[True, False])
    .groupby("stream")
    .head(10)
)

# Display stream, index and Z-score
print(top_10[["stream", "Zscore"]].to_string())

print("\nIndexes of the top 10 Z-scores for each stream:")

for stream, group in top_10.groupby("stream"):
    print(f"\nStream: {stream}")
    print("Indexes:", group.index.tolist())


TOP 10 HIGHEST Z-SCORES FOR EACH STREAM
                        stream  Zscore
320650                       -  1.4685
321372                       -  1.1896
141232                       -  1.1372
78838                        -  1.0902
91190                        -  1.0230
16252                        -   .9825
184931                       -   .9541
154670                       -   .9239
139216                       -   .9079
307420                       -   .8938
26902                     ARTS  2.8319
78460                     ARTS  2.8042
134142                    ARTS  2.7785
211567                    ARTS  2.7784
78994                     ARTS  2.7651
78462                     ARTS  2.6466
151124                    ARTS  2.6336
37590                     ARTS  2.6178
150689                    ARTS  2.6168
84399                     ARTS  2.5970
9891        BIOLOGICAL SCIENCE  3.3182
9958        BIOLOGICAL SCIENCE  3.1195
267926      BIOLOGICAL SCIENCE  3.0710
72109       BIOLOGICAL 

In [12]:
import pandas as pd
import os

# ============================================================
# 1. LOAD DATA
# ============================================================

file_name = "2020_al_data_kaggle_upload_new_old_syllabi.csv"

# Get the folder where this Python file is located
# In Colab, __file__ is not defined. We can assume the file is in the current working directory.
# script_folder = os.path.dirname(os.path.abspath(__file__))

# Create full path to CSV
# Assuming the file is in the default Colab content directory
file_path = os.path.join('/content/', file_name)

data = pd.read_csv(file_path)

print("=" * 60)
print("DATA CLEANSING")
print("=" * 60)

print("\nOriginal rows:", len(data))
print("Original columns:", len(data.columns))


# ============================================================
# 2. REMOVE COMPLETELY EMPTY ROWS
# ============================================================

data = data.dropna(how="all")

print("\nAfter removing completely empty rows:", len(data))


# ============================================================
# 3. REMOVE DUPLICATE RECORDS
# ============================================================

duplicate_count = data.duplicated().sum()

print("\nDuplicate records found:", duplicate_count)

data = data.drop_duplicates()

print("Rows after removing duplicates:", len(data))


# ============================================================
# 4. CLEAN COLUMN NAMES
# ============================================================

data.columns = (
    data.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("\nCleaned column names:")
print(data.columns.tolist())


# ============================================================
# 5. CLEAN TEXT COLUMNS
# ============================================================

text_columns = data.select_dtypes(include=["object"]).columns

for column in text_columns:
    data[column] = data[column].astype("string").str.strip()


# ============================================================
# 6. CONVERT NUMERIC COLUMNS
# ============================================================

numeric_columns = [
    "zscore",
    "district_rank",
    "island_rank",
    "al_year",
    "sub1_r",
    "sub2_r",
    "sub3_r",
    "cgt_r",
    "ge_r",
    "birth_day",
    "birth_year"
]

for column in numeric_columns:

    if column in data.columns:
        data[column] = pd.to_numeric(
            data[column],
            errors="coerce"
        )


# ============================================================
# 7. CLEAN Z-SCORES
# ============================================================

if "zscore" in data.columns:

    invalid_zscore = (
        data["zscore"].notna() &
        ~data["zscore"].between(-5, 5)
    )

    print(
        "\nInvalid Z-score records:",
        invalid_zscore.sum()
    )

    # Replace invalid Z-scores with missing values
    data.loc[invalid_zscore, "zscore"] = pd.NA


# ============================================================
# 8. CLEAN RANK VALUES
# ============================================================

rank_columns = [
    "district_rank",
    "island_rank",
    "sub1_r",
    "sub2_r",
    "sub3_r",
    "cgt_r",
    "ge_r"
]

for column in rank_columns:

    if column in data.columns:

        invalid_rank = (
            data[column].notna() &
            (data[column] <= 0)
        )

        print(
            f"Invalid values in {column}:",
            invalid_rank.sum()
        )

        data.loc[invalid_rank, column] = pd.NA


# ============================================================
# 9. CLEAN BIRTH DAY
# ============================================================

if "birth_day" in data.columns:

    invalid_day = (
        data["birth_day"].notna() &
        ~data["birth_day"].between(1, 31)
    )

    print("\nInvalid birth days:", invalid_day.sum())

    data.loc[invalid_day, "birth_day"] = pd.NA


# ============================================================
# 10. CLEAN BIRTH YEAR
# ============================================================

if "birth_year" in data.columns:

    invalid_year = (
        data["birth_year"].notna() &
        ~data["birth_year"].between(1900, 2020)
    )

    print("Invalid birth years:", invalid_year.sum())

    data.loc[invalid_year, "birth_year"] = pd.NA


# ============================================================
# 11. CLEAN BIRTH MONTH
# ============================================================

if "birth_month" in data.columns:

    # Convert numeric months where possible
    data["birth_month"] = pd.to_numeric(
        data["birth_month"],
        errors="coerce"
    )

    invalid_month = (
        data["birth_month"].notna() &
        ~data["birth_month"].between(1, 12)
    )

    print("Invalid birth months:", invalid_month.sum())

    data.loc[invalid_month, "birth_month"] = pd.NA


# ============================================================
# 12. STANDARDIZE GENDER
# ============================================================

if "gender" in data.columns:

    data["gender"] = (
        data["gender"]
        .str.lower()
        .str.strip()
    )

    data["gender"] = data["gender"].replace({
        "m": "Male",
        "male": "Male",
        "f": "Female",
        "female": "Female"
    })


# ============================================================
# 13. HANDLE MISSING VALUES
# ============================================================

print("\nMissing values after cleaning:")

missing = data.isnull().sum()

print(
    missing[missing > 0]
)


# ============================================================
# 14. SAVE CLEAN DATA
# ============================================================

output_file = os.path.join(
    '/content/',
    "2020_al_data_cleaned.csv"
)

data.to_csv(
    output_file,
    index=False
)

print("\n" + "=" * 60)
print("DATA CLEANSING COMPLETED")
print("=" * 60)

print("\nCleaned rows:", len(data))
print("Cleaned columns:", len(data.columns))

print("\nCleaned file saved to:")
print(output_file)

DATA CLEANSING

Original rows: 337553
Original columns: 19

After removing completely empty rows: 337553

Duplicate records found: 0
Rows after removing duplicates: 337553

Cleaned column names:
['index', 'stream', 'zscore', 'district_rank', 'island_rank', 'al_year', 'sub1', 'sub1_r', 'sub2', 'sub2_r', 'sub3', 'sub3_r', 'cgt_r', 'ge_r', 'syllabus', 'birth_day', 'birth_month', 'birth_year', 'gender']

Invalid Z-score records: 0
Invalid values in district_rank: 0
Invalid values in island_rank: 0
Invalid values in sub1_r: 0
Invalid values in sub2_r: 0
Invalid values in sub3_r: 0
Invalid values in cgt_r: 2
Invalid values in ge_r: 0

Invalid birth days: 0
Invalid birth years: 6
Invalid birth months: 0

Missing values after cleaning:
zscore           105249
district_rank    337553
island_rank      337553
sub1_r           337553
sub2_r           337553
sub3_r           337553
cgt_r             76646
ge_r             337553
birth_day          1583
birth_month      337553
birth_year         158

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [13]:
import pandas as pd
import os

# ============================================================
# 1. LOAD CLEANED DATA
# ============================================================

file_name = "2020_al_data_cleaned.csv"

# In Colab, __file__ is not defined. Assuming the file is in the /content/ directory.
file_path = os.path.join('/content/', file_name)

# Check if the file exists before attempting to load
if not os.path.exists(file_path):
    print(f"Error: The file '{file_name}' was not found at '{file_path}'.")
    print("Please ensure that the data cleansing cell (cell 'YxpM3xdPkOFc') has been executed successfully to create this file.")
else:
    data = pd.read_csv(file_path)

    print("=" * 60)
    print("DATA TRANSFORMATION")
    print("=" * 60)

    print("\nRows:", len(data))
    print("Columns:", len(data.columns))


    # ============================================================
    # 2. STANDARDIZE COLUMN NAMES
    # ============================================================

    data.columns = (
        data.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    print("\nColumn names standardized.")


    # ============================================================
    # 3. STANDARDIZE TEXT VALUES
    # ============================================================

    text_columns = data.select_dtypes(include=["object", "string"]).columns

    for column in text_columns:
        data[column] = (
            data[column]
            .astype("string")
            .str.strip()
        )


    # ============================================================
    # 4. STANDARDIZE GENDER
    # ============================================================

    if "gender" in data.columns:

        data["gender"] = (
            data["gender"]
            .str.lower()
            .str.strip()
        )

        data["gender"] = data["gender"].replace({
            "m": "Male",
            "male": "Male",
            "f": "Female",
            "female": "Female"
        })


    # ============================================================
    # 5. STANDARDIZE STREAM
    # ============================================================

    if "stream" in data.columns:

        data["stream"] = (
            data["stream"]
            .str.strip()
            .str.title()
        )


    # ============================================================
    # 6. CONVERT Z-SCORE TO NUMERIC
    # ============================================================

    if "zscore" in data.columns:

        data["zscore"] = pd.to_numeric(
            data["zscore"],
            errors="coerce"
        )


    # ============================================================
    # 7. CONVERT RANKS TO NUMERIC
    # ============================================================

    rank_columns = [
        "district_rank",
        "island_rank"
    ]

    for column in rank_columns:

        if column in data.columns:

            data[column] = pd.to_numeric(
                data[column],
                errors="coerce"
            )


    # ============================================================
    # 8. CALCULATE TOTAL A PASSES
    # ============================================================

    subject_columns = [
        "sub1",
        "sub2",
        "sub3"
    ]

    available_subjects = [
        column
        for column in subject_columns
        if column in data.columns
    ]

    if len(available_subjects) == 3:

        data["total_a_passes"] = (
            data[available_subjects]
            .apply(
                lambda row: sum(
                    str(value).strip().upper() == "A"
                    for value in row
                ),
                axis=1
            )
        )

    else:

        data["total_a_passes"] = 0


    # ============================================================
    # 9. IDENTIFY STUDENTS WITH 3 A PASSES
    # ============================================================

    data["three_a_student"] = (
        data["total_a_passes"] == 3
    )

    data["three_a_student"] = (
        data["three_a_student"]
        .map({
            True: "Yes",
            False: "No"
        })
    )


    # ============================================================
    # 10. CREATE PERFORMANCE CATEGORY
    # ============================================================

    if "zscore" in data.columns:

        data["performance_category"] = pd.cut(
            data["zscore"],
            bins=[
                float("-inf"),
                0,
                1,
                2,
                float("inf")
            ],
            labels=[
                "Low",
                "Average",
                "Good",
                "Excellent"
            ]
        )


    # ============================================================
    # 11. CREATE RANK CATEGORY
    # ============================================================

    if "island_rank" in data.columns:

        data["rank_category"] = pd.cut(
            data["island_rank"],
            bins=[
                0,
                100,
                1000,
                5000,
                float("inf")
            ],
            labels=[
                "Top 100",
                "Top 1000",
                "Top 5000",
                "Other"
            ]
        )


    # ============================================================
    # 12. CREATE DATE OF BIRTH
    # ============================================================

    if all(
        column in data.columns
        for column in [
            "birth_day",
            "birth_month",
            "birth_year"
        ]
    ):

        data["date_of_birth"] = pd.to_datetime(
            {
                "year": data["birth_year"],
                "month": data["birth_month"],
                "day": data["birth_day"]
            },
            errors="coerce"
        )


    # ============================================================
    # 13. CALCULATE AGE IN 2020
    # ============================================================

    if "date_of_birth" in data.columns:

        data["age_in_2020"] = (
            2020 - data["date_of_birth"].dt.year
        )


    # ============================================================
    # 14. DISPLAY TRANSFORMED DATA
    # ============================================================

    print("\n" + "=" * 60)
    print("TRANSFORMED DATA")
    print("=" * 60)

    print(data.head())


    # ============================================================
    # 15. 3A STUDENT SUMMARY
    # ============================================================

    print("\n" + "=" * 60)
    print("3A STUDENTS BY STREAM")
    print("=" * 60)

    three_a = data[
        data["three_a_student"] == "Yes"
    ]

    if "stream" in data.columns:

        print(
            three_a
            .groupby("stream")
            .size()
        )


    # ============================================================
    # 16. SAVE TRANSFORMED DATA
    # ============================================================

    output_file = os.path.join(
        '/content/',
        "2020_al_data_transformed.csv"
    )

    data.to_csv(
        output_file,
        index=False
    )

    print("\n" + "=" * 60)
    print("TRANSFORMATION COMPLETED")
    print("=" * 60)

    print("\nTransformed file:")
    print(output_file)

DATA TRANSFORMATION

Rows: 337553
Columns: 19

Column names standardized.

TRANSFORMED DATA
   index    stream  zscore  district_rank  island_rank  al_year  \
0      0      Arts -0.3550            NaN          NaN     2020   
1      1      Arts -0.2648            NaN          NaN     2020   
2      2  Commerce -0.4760            NaN          NaN     2020   
3      3  Commerce -0.1012            NaN          NaN     2020   
4      4  Commerce  0.6014            NaN          NaN     2020   

                sub1  sub1_r               sub2  sub2_r  ... birth_day  \
0  POLITICAL SCIENCE     NaN  DANCING(BHARATHA)     NaN  ...      31.0   
1  POLITICAL SCIENCE     NaN     CARNATIC MUSIC     NaN  ...      13.0   
2          ECONOMICS     NaN   BUSINESS STUDIES     NaN  ...      16.0   
3          ECONOMICS     NaN   BUSINESS STUDIES     NaN  ...      16.0   
4          ECONOMICS     NaN   BUSINESS STUDIES     NaN  ...       7.0   

   birth_month  birth_year  gender total_a_passes  three_a_s